# 02 — Tokenizer Training

Trains a byte-pair-encoding (BPE) tokenizer from scratch (Hugging Face
`tokenizers` library — no pretrained tokenizer used) on a mix of:

- All 158 SEC filings collected in `01_data_collection.ipynb`
- A sample of Wikitext-103 (every 5th line, ~20% of the full corpus — kept
  small enough to encode quickly while still giving the tokenizer broad
  general-English coverage, not just SEC-specific vocabulary)

**Vocabulary size: 8,000.** Small by modern LLM standards (GPT-2 uses
50,257) — a deliberate choice given this model's scale (33.5M parameters);
a much larger vocabulary would spend a disproportionate share of the
embedding table on rarely-seen tokens.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q tokenizers
print("Ready.")

Mounted at /content/drive
Ready.

In [1]:
import os
from tokenizers import Tokenizer
from tokenizers.models import BPE
from tokenizers.trainers import BpeTrainer
from tokenizers.pre_tokenizers import Whitespace

# --- Collect all training files ---
data_dir = '/content/drive/MyDrive/sec_chatbot/data'
sec_files = []

for company in os.listdir(data_dir):
    company_path = os.path.join(data_dir, company)
    if os.path.isdir(company_path):
        for f in os.listdir(company_path):
            if f.endswith('.txt'):
                sec_files.append(os.path.join(company_path, f))

print(f"SEC filings to train on: {len(sec_files)}")

SEC filings to train on: 158

In [1]:
# We use a SAMPLE of Wikitext-103, not all 500MB -- too large/slow for
# tokenizer training and not necessary for good subword coverage.
wiki_path = '/content/drive/MyDrive/sec_chatbot/data/wikitext103_train.txt'
wiki_sample_path = '/content/wiki_sample.txt'

print("Sampling Wikitext-103...")
with open(wiki_path, 'r', encoding='utf-8') as f_in, \
     open(wiki_sample_path, 'w', encoding='utf-8') as f_out:
    for i, line in enumerate(f_in):
        if i % 5 == 0:  # every 5th line = ~20% of the dataset
            f_out.write(line)
print("Sample written.")

all_files = sec_files + [wiki_sample_path]
print(f"Total files for tokenizer training: {len(all_files)}")

Sampling Wikitext-103...
Sample written.
Total files for tokenizer training: 159

In [1]:
# --- Build and train the tokenizer ---
# Special tokens are reserved words the model uses for structure:
# <PAD> = padding, <UNK> = unknown/out-of-vocabulary, <BOS>/<EOS> = sequence boundaries
special_tokens = ["<PAD>", "<UNK>", "<BOS>", "<EOS>"]

tokenizer = Tokenizer(BPE(unk_token="<UNK>"))
tokenizer.pre_tokenizer = Whitespace()  # split into words before BPE merges run

trainer = BpeTrainer(
    vocab_size=8000,
    min_frequency=2,      # a pair must appear at least twice to be merged
    special_tokens=special_tokens,
    show_progress=True,
)

print("Training BPE tokenizer...")
tokenizer.train(all_files, trainer)
print(f"Vocabulary size: {tokenizer.get_vocab_size()}")

# --- Save ---
tokenizer_save_dir = '/content/drive/MyDrive/sec_chatbot/tokenizer'
os.makedirs(tokenizer_save_dir, exist_ok=True)
tokenizer.save(os.path.join(tokenizer_save_dir, 'tokenizer.json'))
print(f"Tokenizer saved to {tokenizer_save_dir}")

Training BPE tokenizer...
Vocabulary size: 8000
Tokenizer saved to /content/drive/MyDrive/sec_chatbot/tokenizer

In [1]:
# --- Sanity-check it on real sentences ---
test_sentences = [
    "Apple's revenue grew 8% year over year.",
    "The company faces significant risk from supply chain disruptions.",
    "Operating income increased due to higher gross margins.",
]

for sentence in test_sentences:
    encoded = tokenizer.encode(sentence)
    decoded = tokenizer.decode(encoded.ids)
    print(f"\nInput:   {sentence}")
    print(f"Tokens:  {encoded.tokens}")
    print(f"IDs:     {encoded.ids}")
    print(f"Decoded: {decoded}")


Input:   Apple's revenue grew 8% year over year.
Tokens:  ['Apple', "'", 's', 'revenue', 'grew', '8', '%', 'year', 'over', 'year', '.']
IDs:     [4783, 10, 86, 1737, 5667, 27, 8, 1219, 1162, 1219, 17]
Decoded: Apple ' s revenue grew 8 % year over year .

Input:   The company faces significant risk from supply chain disruptions.
Tokens:  ['The', 'company', 'faces', 'significant', 'risk', 'from', 'supply', 'chain', 'disrup', 'tions', '.']
IDs:     [1028, 1935, 7885, 1775, 3146, 1081, 3153, 4957, 4168, 2883, 17]
Decoded: The company faces significant risk from supply chain disrup tions .

Input:   Operating income increased due to higher gross margins.
Tokens:  ['Operating', 'income', 'increased', 'due', 'to', 'higher', 'gross', 'margins', '.']
IDs:     [3489, 1574, 2116, 1568, 990, 2970, 3865, 7044, 17]
Decoded: Operating income increased due to higher gross margins .